In [2]:
import os
import re
from dotenv import load_dotenv
from langchain_core.documents import Document
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [3]:
load_dotenv()
groq_api = os.getenv('GROQ_API_KEY')


In [18]:
def load_youtube_snippets(url: str, window_seconds: int = 60, overlap_seconds: int = 10):

    match = re.search(r"(?:v=|youtu\.be/)([\w-]+)", url)

    if not match:
        raise ValueError(f"Could not extract video ID from URL: {url}")
    
    video_id = match.group(1)
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en', 'bn', 'hi'])
    snippets = transcript.snippets
    
    chunks = []
    current_text = []
    window_start = snippets[0].start if snippets else 0.0
    last_end = window_start

    for snippet in snippets:
        current_text.append(snippet.text)
        last_end = snippet.start + snippet.duration

        if last_end - window_start >= window_seconds:
            chunks.append(Document(
                page_content=" ".join(current_text).strip(),
                metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)}
            ))
            overlap_start = max(window_start, last_end - overlap_seconds)
            current_text = [s.text for s in snippets if overlap_start <= s.start < last_end]
            window_start = overlap_start

    if current_text:
        chunks.append(Document(
            page_content=" ".join(current_text).strip(),
            metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)}
        ))

    return chunks

texts = load_youtube_snippets("https://www.youtube.com/watch?v=tL9Lw250spc")
